In [27]:
# coding:utf-8
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from load_data import PyTorchTrainDataLoader
import ctypes
import os
import time
import numpy as np
import pandas as pd


## 一、补全关键代码并运行

In [2]:
class Config(object):

    def __init__(self):
        self.p_norm = 1
        self.margin = 1.0
        self.hidden_size = 50
        self.nbatches = 100
        self.entity = 0
        self.relation = 0
        self.trainTimes = 10 #  1   shaw
        self.learningRate = 0.01
        self.use_gpu = True #  False   shaw

def to_var(x, use_gpu):
		if use_gpu:
			return Variable(torch.from_numpy(x).cuda())
		else:
			return Variable(torch.from_numpy(x))

In [3]:
class TransE(nn.Module):

    def __init__(self, ent_tot, rel_tot, dim = 100, p_norm = 1, norm_flag = True, margin = None):
        '''
        Paramters:
        p_norm: 1 for l1-norm, 2 for l2-norm
        norm_flag: if use normalization
        margin: margin in loss function
        '''
        super(TransE, self).__init__()

        self.dim = dim
        self.margin = margin
        self.norm_flag = norm_flag
        self.p_norm = p_norm
        self.ent_tot = ent_tot
        self.rel_tot = rel_tot
        self.ent_embeddings = nn.Embedding(self.ent_tot, self.dim)
        self.rel_embeddings = nn.Embedding(self.rel_tot, self.dim)

        nn.init.xavier_uniform_(self.ent_embeddings.weight.data)
        nn.init.xavier_uniform_(self.rel_embeddings.weight.data)
            
        if margin != None:
            self.margin = nn.Parameter(torch.Tensor([margin]))
            self.margin.requires_grad = False
            self.margin_flag = True
        else:
            self.margin_flag = False


    def _calc(self, h, t, r):
        # TO DO: implement score function
        # Hint: you can use F.normalize and torch.norm functions
        if self.norm_flag: # normalize embeddings with l2 norm
            # 2, batch_size, hidden_dim
            h = F.normalize(h, p=2, dim=2)
            r = F.normalize(r, p=2, dim=2)
            t = F.normalize(t, p=2, dim=2)
        # 2, batch_size
        score = h + r - t
        score = torch.norm(score, self.p_norm, dim=2)
        return score
    
    def forward(self, data):
        batch_h = data['batch_h']
        batch_t = data['batch_t']
        batch_r = data['batch_r']
        h = self.ent_embeddings(batch_h)
        t = self.ent_embeddings(batch_t)
        r = self.rel_embeddings(batch_r)
        score = self._calc(h ,t, r)
        return score

    def predict(self, data):
        score = self.forward(data)
        return score.cpu().data.numpy()
    
    def loss(self, pos_score, neg_score):
        # TO DO: implement loss function
        # Hint: consider margin
        z = self.margin + (pos_score - neg_score).mean()
        l = torch.nn.ReLU()(z)
        return l

In [4]:
def main():
    config = Config()
    train_dataloader = PyTorchTrainDataLoader(
                            in_path = "./data/", 
                            nbatches = config.nbatches,
                            threads = 8)
    
    transe = TransE(
            ent_tot = train_dataloader.get_ent_tot(),
            rel_tot = train_dataloader.get_rel_tot(),
            dim = config.hidden_size, 
            p_norm = config.p_norm, 
            norm_flag = True,
            margin=config.margin)
    
    optimizier = optim.SGD(transe.parameters(), lr=config.learningRate)

    if config.use_gpu:
        transe.cuda()
    
    for times in range(config.trainTimes):
        ep_loss = 0.
        for data in train_dataloader:
            optimizier.zero_grad()
            score = transe({
                    'batch_h': to_var(data['batch_h'], config.use_gpu).long(),
                    'batch_t': to_var(data['batch_t'], config.use_gpu).long(),
                    'batch_r': to_var(data['batch_r'], config.use_gpu).long()})
            pos_score, neg_score = score[0], score[1]
            loss = transe.loss(pos_score, neg_score)
            loss.backward()
            optimizier.step()
            ep_loss += loss.item()
        print("Epoch %d | loss: %f" % (times+1, ep_loss))
    
    print("Finish Training")
    
    f = open("entity2vec.txt", "w")
    enb = transe.ent_embeddings.weight.data.cpu().numpy()
    for i in enb:
        for j in i:
            f.write("%f\t" % (j))
        f.write("\n")
    f.close()

    f = open("relation2vec.txt", "w")
    enb = transe.rel_embeddings.weight.data.cpu().numpy()
    for i in enb:
        for j in i:
            f.write("%f\t" % (j))
        f.write("\n")
    f.close()

In [5]:
if __name__ == "__main__":
    main()


Epoch 1 | loss: 48.658325
Epoch 2 | loss: 13.798743
Epoch 3 | loss: 0.325288
Epoch 4 | loss: 0.078978
Epoch 5 | loss: 0.030715
Epoch 6 | loss: 0.058220
Epoch 7 | loss: 0.002874
Epoch 8 | loss: 0.045722
Epoch 9 | loss: 0.004666
Epoch 10 | loss: 0.000000
Finish Training


## 二、给定头实体Q30，关系P36，最接近的尾实体

In [135]:
# 从结果文件中读取头实体和关系
entity = np.loadtxt('./entity2vec.txt')
entity2id = pd.read_csv('./data/entity2id.txt','\t', header=None, names=['d1','d2'], skiprows=1)
relation = np.loadtxt('./relation2vec.txt')
relation2id = pd.read_csv('./data/relation2id.txt','\t', header=None, names=['d1','d2'], skiprows=1)

/tmp/ipykernel_86973/3144982152.py:3: FutureWarning: In a future version of pandas all arguments of read_csv except for the argument 'filepath_or_buffer' will be keyword-only.
  entity2id = pd.read_csv('./data/entity2id.txt','\t', header=None, names=['d1','d2'], skiprows=1)
/tmp/ipykernel_86973/3144982152.py:5: FutureWarning: In a future version of pandas all arguments of read_csv except for the argument 'filepath_or_buffer' will be keyword-only.
  relation2id = pd.read_csv('./data/relation2id.txt','\t', header=None, names=['d1','d2'], skiprows=1)


In [146]:
q30 = entity[entity2id.query("d1=='Q30'")['d2']]
p36 = relation[relation2id.query("d1=='P36'")['d2']]
t = q30 + p36

In [147]:
stack = {}
for index,ent in enumerate(entity):
    sim = np.linalg.norm(t - ent,2)
    stack[index] = sim

In [148]:
list = sorted(stack.items(), key=lambda x:x[1])
for l in list[:10]:
    print(l[0],entity2id.query("d2=={}".format(l[0]))['d1'])

48828 21031    Q731647
Name: d1, dtype: object
12862 37235    Q490923
Name: d1, dtype: object
39057 46839    Q448362
Name: d1, dtype: object
12988 13439    Q4008388
Name: d1, dtype: object
41609 26107    Q14781036
Name: d1, dtype: object
31187 33745    Q15314958
Name: d1, dtype: object
46024 36089    Q219512
Name: d1, dtype: object
41343 4133    Q14508203
Name: d1, dtype: object
10470 38224    Q909158
Name: d1, dtype: object
49266 6359    Q309589
Name: d1, dtype: object


## 三、给定头实体Q30，尾实体Q49，最接近的关系

In [143]:
q49 = entity[entity2id.query("d1=='Q49'")['d2']]

r = q49 - q30

In [144]:
stack = {}
for index,rel in enumerate(relation):
    sim = np.linalg.norm(r - rel,2)
    stack[index] = sim

In [145]:
list = sorted(stack.items(), key=lambda x:x[1])
for l in list[:10]:
    print(l[0],relation2id.query("d2=={}".format(l[0]))['d1'])

271 373    P427
Name: d1, dtype: object
347 242    P1073
Name: d1, dtype: object
19 317    P30
Name: d1, dtype: object
313 281    P885
Name: d1, dtype: object
0 178    P607
Name: d1, dtype: object
376 254    P2378
Name: d1, dtype: object
161 310    P118
Name: d1, dtype: object
201 336    P1071
Name: d1, dtype: object
355 55    P783
Name: d1, dtype: object
215 129    P579
Name: d1, dtype: object


## 四、参数p_norm和margin对模型的影响